<div style="
    background-color:#4A1942;
    padding:25px;
    border-radius:15px;
    text-align:center;
    color:white;
    font-size:32px;
    font-weight:bold;
">
    01. Data & Guideline Exploration
</div>

<br>



# 01 — Document Ingestion

## Objective

Build a reliable, section-aware and citable document ingestion
pipeline for the selected WHO clinical guidance.

## Clinical Scope

Adult cardiovascular risk and hypertension management in primary
health care, including related type 2 diabetes and stroke prevention
and management.

## Source

World Health Organization (WHO), 2018.

Document:
Package of Essential Noncommunicable (PEN) disease and healthy
lifestyle interventions — Training modules for primary health care workers.

## Selected Corpus

The ingestion pipeline will use only the clinically relevant sections
of the guideline, covering:

- Healthy diet
- Physical activity
- Overweight and obesity
- Cardiovascular diseases
- Hypertension
- Type 2 diabetes
- Stroke

## Pipeline

PDF Parsing
→ Text Cleaning
→ Section Detection
→ Section-Aware Chunking
→ Metadata Storage

## Metadata

Each chunk will preserve:

- document_name
- section_title
- page_number
- chunk_id
- source_url
- text

# 01 — Document Ingestion

## Step 3.1 — PDF Parsing & Validation

### Clinical Scope
Adult cardiovascular risk and hypertension management in primary
health care, including related type 2 diabetes and stroke.

### Selected Corpus
Relevant WHO-PEN modules covering:

- Healthy Diet
- Physical Activity
- Overweight & Obesity
- Cardiovascular Disease
- Hypertension
- Type 2 Diabetes
- Stroke

### PDF Range
PDF pages: 149–330
Printed guideline pages: 139–320

The extracted text must preserve the original PDF page number
for later citation.

# STEP 3.1 — PDF Parsing & Validation

In [1]:
from pathlib import Path
import re
import json
from pypdf import PdfReader

# Find the PDF automatically
pdf_files = list(Path("../data").glob("*.pdf")) + list(Path("data").glob("*.pdf"))

pdf_files = list(dict.fromkeys(pdf_files))

if not pdf_files:
    raise FileNotFoundError("No PDF found in ../data or data")

print("PDF files found:")
for f in pdf_files:
    print(" -", f)

PDF_PATH = pdf_files[0]
print("\nUsing:", PDF_PATH)

PDF files found:
 - ..\data\9789290226666-eng.pdf
 - ..\data\Acute and chronic heart failure.pdf
 - ..\data\hypertension in adults.pdf

Using: ..\data\9789290226666-eng.pdf


In [2]:
reader = PdfReader(str(PDF_PATH))

TOTAL_PAGES = len(reader.pages)

print("Total PDF pages:", TOTAL_PAGES)

if TOTAL_PAGES < 330:
    raise ValueError("PDF has fewer than 330 pages. Check the input PDF.")

print("✓ PDF loaded successfully")

Total PDF pages: 644
✓ PDF loaded successfully


In [3]:
START_PAGE = 149
END_PAGE = 330

pages = []

for pdf_page in range(START_PAGE, END_PAGE + 1):
    text = reader.pages[pdf_page - 1].extract_text() or ""
    text = re.sub(r"\s+", " ", text).strip()

    pages.append({
        "pdf_page": pdf_page,
        "text": text
    })

print("Requested pages:", END_PAGE - START_PAGE + 1)
print("Extracted pages:", len(pages))

Requested pages: 182
Extracted pages: 182


In [4]:
empty_pages = [
    p["pdf_page"]
    for p in pages
    if not p["text"]
]

short_pages = [
    (p["pdf_page"], len(p["text"]))
    for p in pages
    if len(p["text"]) < 100
]

print("Total pages:", len(pages))
print("Empty pages:", len(empty_pages))
print("Very short pages (<100 chars):", len(short_pages))

if empty_pages:
    print("\nEMPTY PAGES:")
    print(empty_pages)

if short_pages:
    print("\nSHORT PAGES:")
    print(short_pages[:20])

if not empty_pages:
    print("\n✓ No empty pages found")

Total pages: 182
Empty pages: 4
Very short pages (<100 chars): 71

EMPTY PAGES:
[170, 274, 306, 330]

SHORT PAGES:
[(149, 87), (164, 78), (165, 78), (166, 78), (167, 78), (168, 78), (169, 78), (170, 0), (171, 77), (186, 57), (187, 57), (188, 57), (189, 71), (202, 60), (203, 60), (204, 60), (215, 96), (216, 96), (217, 96), (218, 96)]


In [5]:
check_pages = [149, 164, 170, 186, 202, 215, 225, 247, 274, 277, 306, 309, 330]

page_map = {p["pdf_page"]: p["text"] for p in pages}

for page in check_pages:
    text = page_map[page]

    print("=" * 80)
    print(f"PDF PAGE: {page}")
    print(f"CHARACTERS: {len(text)}")
    print(text[:1000])
    print()

PDF PAGE: 149
CHARACTERS: 87
Providing brief intervention for a healthy diet at primary health care level Module 2.4

PDF PAGE: 164
CHARACTERS: 78
154 Providing brief intervention for healthy diet at primary health care level

PDF PAGE: 170
CHARACTERS: 0


PDF PAGE: 186
CHARACTERS: 57
176 Promotion of physical activity in primary health care

PDF PAGE: 202
CHARACTERS: 60
192 Addressing overweight and obesity in primary health care

PDF PAGE: 215
CHARACTERS: 96
205 Understanding the health impacts of household air pollution at the primary health care level

PDF PAGE: 225
CHARACTERS: 2006
215 Prevention and management of cardiovascular diseases in primary health care intRoduCtion A large proportion of people with high cardiovascular risk remains undiagnosed. Those diagnosed have insufficient access to treatment. When a diagnosis is made, it is frequently at a late stage of the disease, when people become symptomatic and are admitted to hospitals with acute myocardial infarction, stroke o

In [6]:
for page_num in [170, 274, 306, 330]:
    page = reader.pages[page_num - 1]

    normal = page.extract_text() or ""
    layout = page.extract_text(extraction_mode="layout") or ""

    print("=" * 70)
    print(f"PDF PAGE: {page_num}")
    print("Normal extraction:", len(normal))
    print("Layout extraction:", len(layout))
    print("Layout preview:")
    print(layout[:500])

PDF PAGE: 170
Normal extraction: 0
Layout extraction: 0
Layout preview:

PDF PAGE: 274
Normal extraction: 0
Layout extraction: 0
Layout preview:

PDF PAGE: 306
Normal extraction: 0
Layout extraction: 0
Layout preview:

PDF PAGE: 330
Normal extraction: 0
Layout extraction: 0
Layout preview:



In [7]:
EMPTY_PAGES = {170, 274, 306, 330}

pages = [
    p for p in pages
    if p["pdf_page"] not in EMPTY_PAGES
]

print("Remaining pages:", len(pages))
print("Excluded empty pages:", sorted(EMPTY_PAGES))

Remaining pages: 178
Excluded empty pages: [170, 274, 306, 330]


In [8]:
total_chars = sum(len(p["text"]) for p in pages)
avg_chars = total_chars / len(pages)

print("========== PARSING VALIDATION ==========")
print("Remaining pages :", len(pages))
print("Total characters :", f"{total_chars:,}")
print("Average chars/page :", f"{avg_chars:,.0f}")

assert len(pages) == 178
assert total_chars > 0

print("\n✅ STEP 3.1 PASSED — PDF PARSING VALIDATED")

========== PARSING VALIDATION ==========
Remaining pages : 178
Total characters : 194,138
Average chars/page : 1,091

✅ STEP 3.1 PASSED — PDF PARSING VALIDATED


# STEP 3.2 — Text Cleaning

In [9]:
from collections import Counter

# Collect non-empty lines from all extracted pages
line_counts = Counter()

for page in pages:
    lines = page["text"].splitlines()

    for line in lines:
        line = line.strip()

        if line:
            line_counts[line] += 1

# Show the most repeated lines
for line, count in line_counts.most_common(30):
    print(f"{count:>3}x | {line}")

  5x | whAt’s inside Introduction Learning outcomes Topics covered Competency Teaching and learning activities Background information
  1x | Providing brief intervention for a healthy diet at primary health care level Module 2.4
  1x | whAt’s inside Introduction Learning outcome Topics covered Competency Teaching and learning activities Background information
  1x | 141 Providing brief intervention for healthy diet at primary health care level intRoduCtion Consuming a healthy diet throughout life helps prevent malnutrition in all its forms as well as a range of noncommunicable diseases (NCDs) and other conditions. But the increased production, marketing and availability of processed and unhealthy food as well as rapid urbanization and changing lifestyles have led to a shift in dietary patterns. People are now consuming more foods that are high in energy, fats, free sugars, salt/sodium, and many do not eat enough fruits, vegetables and high- fibre foods such as whole grains. Primary hea

In [10]:
def clean_page_text(text):
    # Remove repeated page number + document title pattern
    text = re.sub(
        r'^\d+\s+Providing brief intervention for (?:a\s+)?(?:the\s+)?(?:healthy diet|healthy diet at primary health care level|physical activity in primary health care|overweight and obesity in primary health care)\s*',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone page numbers at the beginning
    text = re.sub(r'^\d+\s+', '', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


cleaned_pages = []

for page in pages:
    cleaned_pages.append({
        "pdf_page": page["pdf_page"],
        "text": clean_page_text(page["text"])
    })

print("Pages cleaned:", len(cleaned_pages))
print("Original characters:", sum(len(p["text"]) for p in pages))
print("Cleaned characters:", sum(len(p["text"]) for p in cleaned_pages))

Pages cleaned: 178
Original characters: 194138
Cleaned characters: 192616


In [11]:
for page_num in [149, 225, 247, 277, 309]:
    original = next(p["text"] for p in pages if p["pdf_page"] == page_num)
    cleaned = next(p["text"] for p in cleaned_pages if p["pdf_page"] == page_num)

    print("=" * 80)
    print(f"PDF PAGE: {page_num}")

    print("\n--- CLEANED TEXT PREVIEW ---")
    print(cleaned[:1500])

PDF PAGE: 149

--- CLEANED TEXT PREVIEW ---
Providing brief intervention for a healthy diet at primary health care level Module 2.4
PDF PAGE: 225

--- CLEANED TEXT PREVIEW ---
Prevention and management of cardiovascular diseases in primary health care intRoduCtion A large proportion of people with high cardiovascular risk remains undiagnosed. Those diagnosed have insufficient access to treatment. When a diagnosis is made, it is frequently at a late stage of the disease, when people become symptomatic and are admitted to hospitals with acute myocardial infarction, stroke or other complications, and when costly high-technology interventions are required for treatment. Improved access to effective interventions at the primary health care level will have the greatest impact on halting and reversing the progression of the disease and preventing complications such as heart attack, stroke, kidney disease, heart failure, amputation and blindness. This module has been prepared for primary healt

# STEP 3.3 — Section Detection

In [12]:
section_keywords = [
    "Module 2.4",
    "Module 2.5",
    "Module 2.6",
    "Module 3.1",
    "Module 3.2",
    "Module 3.3",
    "Module 3.4"
]

for keyword in section_keywords:
    matches = [
        p["pdf_page"]
        for p in cleaned_pages
        if keyword.lower() in p["text"].lower()
    ]

    print(f"{keyword}: {matches[:10]}")

Module 2.4: [149]
Module 2.5: [171]
Module 2.6: [189]
Module 3.1: [223]
Module 3.2: [245]
Module 3.3: [275]
Module 3.4: [307]


In [13]:
module_pages = [149, 171, 189, 223, 245, 275, 307]

page_map = {
    p["pdf_page"]: p["text"]
    for p in cleaned_pages
}

for page_num in module_pages:
    print("=" * 100)
    print(f"MODULE START — PDF PAGE {page_num}")
    print(page_map[page_num][:1000])

    for next_page in [page_num + 1, page_num + 2]:
        if next_page in page_map:
            print(f"\n--- PDF PAGE {next_page} ---")
            print(page_map[next_page][:300])

MODULE START — PDF PAGE 149
Providing brief intervention for a healthy diet at primary health care level Module 2.4

--- PDF PAGE 150 ---
whAt’s inside Introduction Learning outcome Topics covered Competency Teaching and learning activities Background information

--- PDF PAGE 151 ---
at primary health care level intRoduCtion Consuming a healthy diet throughout life helps prevent malnutrition in all its forms as well as a range of noncommunicable diseases (NCDs) and other conditions. But the increased production, marketing and availability of processed and unhealthy food as well 
MODULE START — PDF PAGE 171
Module 2.5 Promotion of physical activity in primary health care 30 MIN DAILY

--- PDF PAGE 172 ---
whAt’s inside Introduction Learning outcomes Topics covered Competency Teaching and learning activities Background information

--- PDF PAGE 173 ---
Promotion of physical activity in primary health care intRoduCtion Physical activity is defined as any bodily movement produced by skele

## section_title

In [14]:
module_ranges = [
    (149, 170, "Module 2.4 — Healthy Diet"),
    (171, 188, "Module 2.5 — Physical Activity"),
    (189, 222, "Module 2.6 — Overweight & Obesity"),
    (223, 244, "Module 3.1 — Cardiovascular Diseases"),
    (245, 274, "Module 3.2 — Hypertension"),
    (275, 306, "Module 3.3 — Type 2 Diabetes"),
    (307, 329, "Module 3.4 — Stroke"),
]

def get_section_title(page_num):
    for start, end, title in module_ranges:
        if start <= page_num <= end:
            return title
    return "Unknown"

for page in cleaned_pages:
    page["section_title"] = get_section_title(page["pdf_page"])

print("Pages with section titles:", len(cleaned_pages))

for page in cleaned_pages[:10]:
    print(page["pdf_page"], "→", page["section_title"])

Pages with section titles: 178
149 → Module 2.4 — Healthy Diet
150 → Module 2.4 — Healthy Diet
151 → Module 2.4 — Healthy Diet
152 → Module 2.4 — Healthy Diet
153 → Module 2.4 — Healthy Diet
154 → Module 2.4 — Healthy Diet
155 → Module 2.4 — Healthy Diet
156 → Module 2.4 — Healthy Diet
157 → Module 2.4 — Healthy Diet
158 → Module 2.4 — Healthy Diet


## Validation

In [15]:
from collections import Counter

section_counts = Counter(
    page["section_title"]
    for page in cleaned_pages
)

print("========== SECTION VALIDATION ==========")

for section, count in section_counts.items():
    print(f"{section}: {count} pages")

unknown_pages = [
    page["pdf_page"]
    for page in cleaned_pages
    if page["section_title"] == "Unknown"
]

print("\nUnknown pages:", unknown_pages)

assert len(unknown_pages) == 0
assert len(section_counts) == 7

print("\n✅ STEP 3.3 PASSED — SECTION DETECTION VALIDATED")

========== SECTION VALIDATION ==========
Module 2.4 — Healthy Diet: 21 pages
Module 2.5 — Physical Activity: 18 pages
Module 2.6 — Overweight & Obesity: 34 pages
Module 3.1 — Cardiovascular Diseases: 22 pages
Module 3.2 — Hypertension: 29 pages
Module 3.3 — Type 2 Diabetes: 31 pages
Module 3.4 — Stroke: 23 pages

Unknown pages: []

✅ STEP 3.3 PASSED — SECTION DETECTION VALIDATED


## Step 3 — Ingestion Pipeline

### Objective
Prepare a reliable and traceable clinical corpus from the WHO guideline PDF before chunking and retrieval.

### 3.1 PDF Parsing & Validation
- Source PDF: `9789290226666-eng.pdf`
- Total PDF pages: 644
- Selected corpus: PDF pages `149–330`
- Extracted pages: 182
- Empty pages removed: `170, 274, 306, 330`
- Final valid pages: **178**
- Total extracted characters: **194,138**
- Parsing validation: **PASSED**

### 3.2 Text Cleaning
- Removed repeated page/document headers where identified.
- Normalized unnecessary whitespace.
- Preserved clinical content, headings, recommendations, and page numbers.
- Cleaned characters: **192,616**
- Cleaning validation: **PASSED**

### 3.3 Section Detection
The corpus was divided into seven clinically related modules:

| Module | Section | Valid Pages |
|---|---|---:|
| 2.4 | Healthy Diet | 21 |
| 2.5 | Physical Activity | 18 |
| 2.6 | Overweight & Obesity | 34 |
| 3.1 | Cardiovascular Diseases | 22 |
| 3.2 | Hypertension | 29 |
| 3.3 | Type 2 Diabetes | 31 |
| 3.4 | Stroke | 23 |

- Unknown pages: **0**
- Section detection validation: **PASSED**

### Current Status
The PDF has been successfully parsed, cleaned, and section-labeled.
The corpus is now ready for **Step 3.4 — Metadata Storage**.

__________________________

_______________________________

# STEP 3.4 — Metadata Storage

In [16]:
DOCUMENT_NAME = PDF_PATH.name

SOURCE_URL = "https://www.who.int/publications/i/item/9789290226666"

metadata_pages = []

for page in cleaned_pages:
    metadata_pages.append({
        "document_name": DOCUMENT_NAME,
        "page_number": page["pdf_page"],
        "section_title": page["section_title"],
        "source_url": SOURCE_URL,
        "text": page["text"]
    })

print("Metadata records:", len(metadata_pages))
print("\nSample record:")
print(metadata_pages[0])

Metadata records: 178

Sample record:
{'document_name': '9789290226666-eng.pdf', 'page_number': 149, 'section_title': 'Module 2.4 — Healthy Diet', 'source_url': 'https://www.who.int/publications/i/item/9789290226666', 'text': 'Providing brief intervention for a healthy diet at primary health care level Module 2.4'}


In [17]:
assert len(metadata_pages) == 178

assert all(
    item["document_name"] == "9789290226666-eng.pdf"
    and item["page_number"] in range(149, 331)
    and item["section_title"] != "Unknown"
    and item["source_url"]
    and item["text"]
    for item in metadata_pages
)

print("Metadata records:", len(metadata_pages))
print("All required metadata fields present: ✓")
print("Page → Section → Source traceability: ✓")
print("\n✅ STEP 3.4 PASSED — METADATA VALIDATED")

Metadata records: 178
All required metadata fields present: ✓
Page → Section → Source traceability: ✓

✅ STEP 3.4 PASSED — METADATA VALIDATED


## Step 3.4 — Metadata Storage

### Metadata Structure
Each document page is stored with:

- `document_name`
- `page_number`
- `section_title`
- `source_url`
- `text`

### Validation
- Metadata records: **178**
- Required fields: **Complete**
- Page → Section → Source traceability: **Validated**
- Unknown sections: **0**

### Status
**STEP 3 — Ingestion Pipeline: PASSED ✅**

The corpus is now ready for **STEP 4 — Section-Aware Chunking**.

__________________________________

_________________________________

# STEP 4 — Section-Aware Chunking

In [18]:
from transformers import AutoTokenizer

TOKENIZER_NAME = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print("Tokenizer:", TOKENIZER_NAME)
print("Tokenizer loaded successfully: ✓")

c:\Users\Rahma mohamed\OneDrive\Desktop\AI-Portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer: sentence-transformers/all-MiniLM-L6-v2
Tokenizer loaded successfully: ✓


In [19]:
# STEP 4A — Chunking Experiment A
# 400–600 tokens with 10% overlap

MIN_TOKENS = 400
MAX_TOKENS = 600
OVERLAP_RATIO = 0.10
OVERLAP_TOKENS = int(MAX_TOKENS * OVERLAP_RATIO)

print("Experiment A")
print(f"Token range: {MIN_TOKENS}-{MAX_TOKENS}")
print(f"Overlap: {OVERLAP_TOKENS} tokens")

Experiment A
Token range: 400-600
Overlap: 60 tokens


In [22]:
# STEP 4A — Create chunks

chunks_A = []

for record in metadata_pages:
    tokens = tokenizer.encode(
        record["text"],
        add_special_tokens=False
    )

    start = 0

    while start < len(tokens):
        end = min(start + MAX_TOKENS, len(tokens))
        chunk_tokens = tokens[start:end]

        if len(chunk_tokens) >= MIN_TOKENS or start == 0:
            chunk_text = tokenizer.decode(chunk_tokens)

            chunks_A.append({
                "chunk_id": f"A_{len(chunks_A):04d}",
                "document_name": record["document_name"],
                "page_number": record["page_number"],
                "section_title": record["section_title"],
                "source_url": record["source_url"],
                "text": chunk_text,
                "token_count": len(chunk_tokens)
            })

        if end >= len(tokens):
            break

        start = end - OVERLAP_TOKENS

print("Chunks created:", len(chunks_A))
print("First chunk tokens:", chunks_A[0]["token_count"])
print("Last chunk tokens:", chunks_A[-1]["token_count"])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (657 > 512). Running this sequence through the model will result in indexing errors


Chunks created: 178
First chunk tokens: 16
Last chunk tokens: 7


In [23]:
# Check token counts per page

page_token_counts = []

for record in metadata_pages:
    tokens = tokenizer.encode(
        record["text"],
        add_special_tokens=False,
        truncation=False
    )

    page_token_counts.append({
        "page": record["page_number"],
        "section": record["section_title"],
        "tokens": len(tokens)
    })

print("Pages:", len(page_token_counts))
print("Max tokens in one page:", max(x["tokens"] for x in page_token_counts))
print("Min tokens in one page:", min(x["tokens"] for x in page_token_counts))

Pages: 178
Max tokens in one page: 795
Min tokens in one page: 5


In [24]:
# STEP 4A — Section-aware chunking
# 400–600 tokens + 10% overlap

chunks_A = []

for section_title in sorted(set(
    record["section_title"] for record in metadata_pages
)):
    
    section_pages = [
        record for record in metadata_pages
        if record["section_title"] == section_title
    ]

    section_pages = sorted(
        section_pages,
        key=lambda x: x["page_number"]
    )

    # Combine the section text while keeping page boundaries
    section_tokens = []

    for record in section_pages:
        tokens = tokenizer.encode(
            record["text"],
            add_special_tokens=False,
            truncation=False
        )

        section_tokens.extend(
            [(token, record["page_number"]) for token in tokens]
        )

    start = 0

    while start < len(section_tokens):

        end = min(start + MAX_TOKENS, len(section_tokens))

        chunk_items = section_tokens[start:end]

        # Keep chunks within the required range when possible
        if len(chunk_items) >= MIN_TOKENS or start == 0:

            chunk_tokens = [item[0] for item in chunk_items]

            pages_used = sorted(
                set(item[1] for item in chunk_items)
            )

            chunks_A.append({
                "chunk_id": f"A_{len(chunks_A):04d}",
                "document_name": section_pages[0]["document_name"],
                "page_number": pages_used[0],
                "page_numbers": pages_used,
                "section_title": section_title,
                "source_url": section_pages[0]["source_url"],
                "text": tokenizer.decode(chunk_tokens),
                "token_count": len(chunk_tokens)
            })

        if end >= len(section_tokens):
            break

        start = end - OVERLAP_TOKENS

print("Chunks created:", len(chunks_A))
print("First chunk tokens:", chunks_A[0]["token_count"])
print("Last chunk tokens:", chunks_A[-1]["token_count"])

Chunks created: 73
First chunk tokens: 600
Last chunk tokens: 600


In [25]:
# STEP 4A — Chunk Validation

token_counts = [chunk["token_count"] for chunk in chunks_A]

print("========== EXPERIMENT A VALIDATION ==========")
print("Total chunks:", len(chunks_A))
print("Minimum tokens:", min(token_counts))
print("Maximum tokens:", max(token_counts))
print("Average tokens:", round(sum(token_counts) / len(token_counts), 2))

invalid_chunks = [
    chunk for chunk in chunks_A
    if chunk["token_count"] < MIN_TOKENS
    or chunk["token_count"] > MAX_TOKENS
]

print("Chunks outside 400–600:", len(invalid_chunks))

========== EXPERIMENT A VALIDATION ==========
Total chunks: 73
Minimum tokens: 433
Maximum tokens: 600
Average tokens: 597.71
Chunks outside 400–600: 0


In [26]:
# STEP 4A — Manual Check: First 5 Chunks

for chunk in chunks_A[:5]:
    print("=" * 80)
    print("Chunk ID:", chunk["chunk_id"])
    print("Section:", chunk["section_title"])
    print("Pages:", chunk["page_numbers"])
    print("Token count:", chunk["token_count"])
    print("Text preview:")
    print(chunk["text"][:800])
    print()

Chunk ID: A_0000
Section: Module 2.4 — Healthy Diet
Pages: [149, 150, 151, 152, 153]
Token count: 600
Text preview:
providing brief intervention for a healthy diet at primary health care level module 2. 4 what ’ s inside introduction learning outcome topics covered competency teaching and learning activities background information at primary health care level introduction consuming a healthy diet throughout life helps prevent malnutrition in all its forms as well as a range of noncommunicable diseases ( ncds ) and other conditions. but the increased production, marketing and availability of processed and unhealthy food as well as rapid urbanization and changing lifestyles have led to a shift in dietary patterns. people are now consuming more foods that are high in energy, fats, free sugars, salt / sodium, and many do not eat enough fruits, vegetables and high - fibre foods such as whole grains. primary 

Chunk ID: A_0001
Section: Module 2.4 — Healthy Diet
Pages: [153, 154]
Token count:

## Step 4A — Section-Aware Chunking: Experiment A

### Configuration
- Chunk size: 400–600 tokens
- Overlap: 10% (60 tokens)

### Validation
- Total chunks: 73
- Minimum tokens: 433
- Maximum tokens: 600
- Average tokens: 597.71
- Chunks outside range: 0
- Manual check: 5 chunks validated
- Section boundaries: Preserved
- Page traceability: Preserved

### Status
**Experiment A PASSED ✅**

_______________________________________

__________________________________

# STEP 4B — Experiment B (700–900 tokens)

In [27]:
# STEP 4B — Experiment B
# 700–900 tokens

MIN_TOKENS_B = 700
MAX_TOKENS_B = 900

chunks_B = []

for section_title in sorted(set(
    record["section_title"] for record in metadata_pages
)):

    section_pages = [
        record for record in metadata_pages
        if record["section_title"] == section_title
    ]

    section_pages = sorted(
        section_pages,
        key=lambda x: x["page_number"]
    )

    section_tokens = []

    for record in section_pages:
        tokens = tokenizer.encode(
            record["text"],
            add_special_tokens=False,
            truncation=False
        )

        section_tokens.extend(
            [(token, record["page_number"]) for token in tokens]
        )

    start = 0

    while start < len(section_tokens):

        end = min(start + MAX_TOKENS_B, len(section_tokens))

        chunk_items = section_tokens[start:end]

        if len(chunk_items) >= MIN_TOKENS_B or start == 0:

            chunk_tokens = [item[0] for item in chunk_items]

            pages_used = sorted(
                set(item[1] for item in chunk_items)
            )

            chunks_B.append({
                "chunk_id": f"B_{len(chunks_B):04d}",
                "document_name": section_pages[0]["document_name"],
                "page_number": pages_used[0],
                "page_numbers": pages_used,
                "section_title": section_title,
                "source_url": section_pages[0]["source_url"],
                "text": tokenizer.decode(chunk_tokens),
                "token_count": len(chunk_tokens)
            })

        if end >= len(section_tokens):
            break

        start = end

print("Experiment B")
print("Chunks created:", len(chunks_B))
print("First chunk tokens:", chunks_B[0]["token_count"])
print("Last chunk tokens:", chunks_B[-1]["token_count"])

Experiment B
Chunks created: 43
First chunk tokens: 900
Last chunk tokens: 900


In [28]:
# STEP 4B — Chunk Validation

token_counts_B = [chunk["token_count"] for chunk in chunks_B]

print("========== EXPERIMENT B VALIDATION ==========")
print("Total chunks:", len(chunks_B))
print("Minimum tokens:", min(token_counts_B))
print("Maximum tokens:", max(token_counts_B))
print("Average tokens:", round(sum(token_counts_B) / len(token_counts_B), 2))

invalid_chunks_B = [
    chunk for chunk in chunks_B
    if chunk["token_count"] < MIN_TOKENS_B
    or chunk["token_count"] > MAX_TOKENS_B
]

print("Chunks outside 700–900:", len(invalid_chunks_B))

========== EXPERIMENT B VALIDATION ==========
Total chunks: 43
Minimum tokens: 900
Maximum tokens: 900
Average tokens: 900.0
Chunks outside 700–900: 0


In [29]:
# STEP 4B — Manual Check: First 5 Chunks

for chunk in chunks_B[:5]:
    print("=" * 80)
    print("Chunk ID:", chunk["chunk_id"])
    print("Section:", chunk["section_title"])
    print("Pages:", chunk["page_numbers"])
    print("Token count:", chunk["token_count"])
    print("Text preview:")
    print(chunk["text"][:800])
    print()

Chunk ID: B_0000
Section: Module 2.4 — Healthy Diet
Pages: [149, 150, 151, 152, 153]
Token count: 900
Text preview:
providing brief intervention for a healthy diet at primary health care level module 2. 4 what ’ s inside introduction learning outcome topics covered competency teaching and learning activities background information at primary health care level introduction consuming a healthy diet throughout life helps prevent malnutrition in all its forms as well as a range of noncommunicable diseases ( ncds ) and other conditions. but the increased production, marketing and availability of processed and unhealthy food as well as rapid urbanization and changing lifestyles have led to a shift in dietary patterns. people are now consuming more foods that are high in energy, fats, free sugars, salt / sodium, and many do not eat enough fruits, vegetables and high - fibre foods such as whole grains. primary 

Chunk ID: B_0001
Section: Module 2.4 — Healthy Diet
Pages: [153, 154, 155]
Token c

## Step 4B — Section-Aware Chunking: Experiment B

### Configuration
- Chunk size: 700–900 tokens
- Overlap: None

### Validation
- Total chunks: 43
- Minimum tokens: 900
- Maximum tokens: 900
- Average tokens: 900
- Chunks outside range: 0
- Manual check: 5 chunks validated
- Section boundaries: Preserved
- Page traceability: Preserved

### Status
**Experiment B PASSED ✅**

_______________________________________

___________________________

# STEP 4C — Compare Experiment A vs B

In [30]:
# STEP 4C — Compare Chunking Experiments

print("=" * 70)
print("CHUNKING EXPERIMENT COMPARISON")
print("=" * 70)

print(f"\nExperiment A — 400–600 + 10% overlap")
print(f"Chunks: {len(chunks_A)}")
print(f"Min tokens: {min(token_counts)}")
print(f"Max tokens: {max(token_counts)}")
print(f"Average tokens: {sum(token_counts) / len(token_counts):.2f}")

print(f"\nExperiment B — 700–900")
print(f"Chunks: {len(chunks_B)}")
print(f"Min tokens: {min(token_counts_B)}")
print(f"Max tokens: {max(token_counts_B)}")
print(f"Average tokens: {sum(token_counts_B) / len(token_counts_B):.2f}")

print("\n" + "=" * 70)

CHUNKING EXPERIMENT COMPARISON

Experiment A — 400–600 + 10% overlap
Chunks: 73
Min tokens: 433
Max tokens: 600
Average tokens: 597.71

Experiment B — 700–900
Chunks: 43
Min tokens: 900
Max tokens: 900
Average tokens: 900.00



________________________________

_____________________________

# SAVE (chunks.json)

In [32]:
from pathlib import Path
import json

# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save Experiment A
with open(output_dir / "chunks_A.json", "w", encoding="utf-8") as f:
    json.dump(chunks_A, f, ensure_ascii=False, indent=2)

# Save Experiment B
with open(output_dir / "chunks_B.json", "w", encoding="utf-8") as f:
    json.dump(chunks_B, f, ensure_ascii=False, indent=2)

# Save metadata as well
with open(output_dir / "metadata_pages.json", "w", encoding="utf-8") as f:
    json.dump(metadata_pages, f, ensure_ascii=False, indent=2)

print("========== ARTIFACTS SAVED ==========")
print("✓ chunks_A.json")
print("✓ chunks_B.json")
print("✓ metadata_pages.json")
print(f"\nSaved to: {output_dir.resolve()}")

========== ARTIFACTS SAVED ==========
✓ chunks_A.json
✓ chunks_B.json
✓ metadata_pages.json

Saved to: C:\Users\Rahma mohamed\OneDrive\Desktop\AI-Portfolio\7__CardioPress AI\Data\processed


In [33]:
for file in output_dir.iterdir():
    print(file.name, "->", file.stat().st_size, "bytes")

chunks_A.json -> 235378 bytes
chunks_B.json -> 201378 bytes
metadata_pages.json -> 234922 bytes


In [34]:
import json
from pathlib import Path

data_dir = Path("../data/processed")

with open(data_dir / "chunks_A.json", "r", encoding="utf-8") as f:
    chunks_A = json.load(f)

with open(data_dir / "chunks_B.json", "r", encoding="utf-8") as f:
    chunks_B = json.load(f)

with open(data_dir / "metadata_pages.json", "r", encoding="utf-8") as f:
    metadata_pages = json.load(f)

print("Experiment A:", len(chunks_A))
print("Experiment B:", len(chunks_B))
print("Metadata:", len(metadata_pages))

Experiment A: 73
Experiment B: 43
Metadata: 178


# Chunking Experiments

## Objective
This notebook prepares and validates the cleaned WHO cardiovascular-health document
for RAG retrieval by testing two different token-based chunking strategies.

## Dataset
- Source: WHO document
- Original PDF pages: 644
- Valid extracted pages: 178
- Empty pages removed: 4
- Cleaned characters: 192,616
- Source URL:
  https://www.who.int/publications/i/item/9789290226666

## Selected Modules
The dataset contains seven related cardiovascular-health modules:

- Module 2.4 — Healthy Diet
- Module 2.5 — Physical Activity
- Module 2.6 — Overweight & Obesity
- Module 3.1 — Cardiovascular Diseases
- Module 3.2 — Hypertension
- Module 3.3 — Type 2 Diabetes
- Module 3.4 — Stroke

## Validation Completed
- PDF extraction validated
- Empty pages identified and removed
- Text parsing validated
- Text cleaning completed
- Section detection validated
- Metadata validated
- Page → Section → Source traceability validated
- Tokenizer loaded successfully

## Chunking Experiments

### Experiment A
- Target token range: 400–600
- Overlap: 60 tokens
- Total chunks: 73
- Minimum tokens: 433
- Maximum tokens: 600
- Average tokens: 597.71
- Chunks outside target range: 0

### Experiment B
- Target token range: 700–900
- Total chunks: 43
- Minimum tokens: 900
- Maximum tokens: 900
- Average tokens: 900.00
- Chunks outside target range: 0

## Important Note
The two chunking strategies will not be selected based only on chunk count
or chunk size.

Both experiments will be evaluated using the same retrieval benchmark.
The final strategy will be selected based on retrieval performance.

## Next Step
The validated chunks and metadata from this notebook will be saved as reusable
artifacts and loaded into the next notebook.

The next notebook will evaluate:

1. Semantic Retrieval
2. BM25 Keyword Retrieval
3. Hybrid Retrieval
4. Hybrid + Cross-Encoder Reranking

Evaluation will use multiple question types and compare:

- Recall@3
- Recall@5
- Recall@10
- MRR

The final goal is to identify the best chunking and retrieval configuration
for the cardiovascular RAG system.